# Gold 04 - Race Performance Analytics

**Business scenario:** Produce race-level and driver-level business KPIs by combining
race results, qualifying, pit stops and lap times.

**Gold concepts:** Joins, aggregations, conditional aggregation and CASE/business classification.

**Note:** Silver already performed cleansing. Gold only derives business metrics.

In [0]:
CATALOG = "formula1_dev"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

RESULTS = f"{CATALOG}.{SILVER_SCHEMA}.results"
RACES = f"{CATALOG}.{SILVER_SCHEMA}.races"
DRIVERS = f"{CATALOG}.{SILVER_SCHEMA}.drivers"
CONSTRUCTORS = f"{CATALOG}.{SILVER_SCHEMA}.constructors"
QUALIFYING = f"{CATALOG}.{SILVER_SCHEMA}.qualifying"
PIT_STOPS = f"{CATALOG}.{SILVER_SCHEMA}.pit_stops"
LAP_TIMES = f"{CATALOG}.{SILVER_SCHEMA}.lap_times"
CIRCUITS = f"{CATALOG}.{SILVER_SCHEMA}.circuits"

GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.f1_race_performance"

DEMO_MODE = True
DEMO_YEAR = 2021
# DEMO_RACES = 3
# DEMO_DRIVERS = 5

In [0]:
from pyspark.sql import functions as F

results = spark.table(RESULTS)
races = spark.table(RACES)
drivers = spark.table(DRIVERS)
constructors = spark.table(CONSTRUCTORS)
qualifying = spark.table(QUALIFYING)
pit_stops = spark.table(PIT_STOPS)
lap_times = spark.table(LAP_TIMES)
circuits = spark.table(CIRCUITS)

if DEMO_MODE:
    demo_races = (
        races
        .filter(F.col("race_year") == DEMO_YEAR)
        .select("race_id", "race_year", "round", "race_name_clean", "circuit_id", "date")
        .orderBy("round")
        # .limit(DEMO_RACES)
    )

    demo_driver_ids = (
        results.join(demo_races.select("race_id"), "race_id", "inner")
        .select("driver_id")
        .distinct()
        .orderBy("driver_id")
        # .limit(DEMO_DRIVERS)
    )

    results = (
        results.join(demo_races.select("race_id"), "race_id", "inner")
        .join(demo_driver_ids, "driver_id", "inner")
    )
    qualifying = (
        qualifying.join(demo_races.select("race_id"), "race_id", "inner")
        .join(demo_driver_ids, "driver_id", "inner")
    )
    pit_stops = (
        pit_stops.join(demo_races.select("race_id"), "race_id", "inner")
        .join(demo_driver_ids, "driver_id", "inner")
    )
    lap_times = (
        lap_times.join(demo_races.select("race_id"), "race_id", "inner")
        .join(demo_driver_ids, "driver_id", "inner")
    )
    races = demo_races

In [0]:
# 1) Aggregate pit stops before joining.
pit_summary = (
    pit_stops
    .groupBy("race_id", "driver_id")
    .agg(
        F.count("*").alias("total_pit_stops"),
        F.avg("milliseconds_clean").alias("avg_pit_duration_ms"),
        F.min("milliseconds_clean").alias("fastest_pit_duration_ms")
    )
)

# 2) Aggregate lap times before joining.
lap_summary = (
    lap_times
    .groupBy("race_id", "driver_id")
    .agg(
        F.countDistinct("lap").alias("laps_recorded"),
        F.min("milliseconds_clean").alias("fastest_lap_ms"),
        F.avg("milliseconds_clean").alias("average_lap_ms")
    )
)

# 3) Aggregate qualifying to one row per race/driver.
qualifying_summary = (
    qualifying
    .groupBy("race_id", "driver_id")
    .agg(
        F.min("position_int").alias("qualifying_position")
    )
)

In [0]:
# Main driver-race business dataset.
race_driver = (
    results.alias("res")
    .join(
        races.select(
            "race_id", "race_year", "round", "race_name_clean",
            "circuit_id", "date"
        ).alias("r"),
        F.col("res.race_id") == F.col("r.race_id"),
        "inner"
    )
    .join(
        drivers.select("driver_id", "driver_full_name").alias("d"),
        F.col("res.driver_id") == F.col("d.driver_id"),
        "inner"
    )
    .join(
        constructors.select(
            "constructor_id", "constructor_name_clean"
        ).alias("c"),
        F.col("res.constructor_id") == F.col("c.constructor_id"),
        "left"
    )
    .join(
        circuits.select(
            "circuit_id", "circuit_name_clean", "country_clean"
        ).alias("ci"),
        F.col("r.circuit_id") == F.col("ci.circuit_id"),
        "left"
    )
    .join(
        qualifying_summary.alias("q"),
        (F.col("res.race_id") == F.col("q.race_id")) &
        (F.col("res.driver_id") == F.col("q.driver_id")),
        "left"
    )
    .join(
        pit_summary.alias("p"),
        (F.col("res.race_id") == F.col("p.race_id")) &
        (F.col("res.driver_id") == F.col("p.driver_id")),
        "left"
    )
    .join(
        lap_summary.alias("l"),
        (F.col("res.race_id") == F.col("l.race_id")) &
        (F.col("res.driver_id") == F.col("l.driver_id")),
        "left"
    )
    .select(
        F.col("r.race_year"),
        F.col("r.round").alias("race_round"),
        F.col("r.race_id"),
        F.col("r.race_name_clean").alias("race_name"),
        F.col("r.date").alias("race_date"),
        F.col("ci.circuit_name_clean").alias("circuit_name"),
        F.col("ci.country_clean").alias("circuit_country"),
        F.col("res.driver_id"),
        F.col("d.driver_full_name").alias("driver_name"),
        F.col("res.constructor_id"),
        F.col("c.constructor_name_clean").alias("constructor_name"),
        F.col("q.qualifying_position"),
        F.col("res.grid"),
        F.col("res.position").alias("finish_position"),
        F.col("res.points"),
        F.col("res.laps"),
        F.col("p.total_pit_stops"),
        F.col("p.avg_pit_duration_ms"),
        F.col("p.fastest_pit_duration_ms"),
        F.col("l.fastest_lap_ms"),
        F.col("l.average_lap_ms")
    )
)

# Business classification only.
race_driver = race_driver.withColumn(
    "finish_category",
    F.when(F.col("finish_position") == 1, "Winner")
     .when(F.col("finish_position") <= 3, "Podium")
     .when(F.col("finish_position") <= 10, "Points Finisher")
     .otherwise("Outside Points")
)

race_driver = race_driver.withColumn(
    "qualifying_to_finish_change",
    F.col("qualifying_position") - F.col("finish_position")
)

display(race_driver.orderBy("race_year", "race_round", "finish_position"))

In [0]:
(
    race_driver
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

In [0]:
display(
    spark.table(GOLD_TABLE)
    .orderBy("race_year", "race_round", "finish_position")
)

In [0]:
gold_df = spark.table(GOLD_TABLE)
print("Gold row count:", gold_df.count())
print(
    "Duplicate race-driver rows:",
    gold_df.groupBy("race_id", "driver_id").count()
           .filter(F.col("count") > 1).count()
)